In [1]:
!pip install transformers peft bitsandbytes accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00


In [2]:
import transformers
import peft
import bitsandbytes
import accelerate

print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("accelerate:", accelerate.__version__)

transformers: 5.0.0
peft: 0.18.1
bitsandbytes: 0.49.2
accelerate: 1.12.0


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Base model loaded in FP16!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Base model loaded in FP16!


In [6]:
from peft import PeftModel

model_with_adapters = PeftModel.from_pretrained(
    base_model,
    "/content"
)

print("Adapters loaded successfully!")

Adapters loaded successfully!


In [7]:
merged_model = model_with_adapters.merge_and_unload()
print("Model merged successfully!")

Model merged successfully!


In [8]:
import os
os.makedirs("/content/merged_model", exist_ok=True)

merged_model.save_pretrained("/content/merged_model")
tokenizer.save_pretrained("/content/merged_model")

print("Merged model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved!


In [9]:
from transformers import BitsAndBytesConfig

bnb_int8 = BitsAndBytesConfig(load_in_8bit=True)

model_int8 = AutoModelForCausalLM.from_pretrained(
    "/content/merged_model",
    quantization_config=bnb_int8,
    device_map="auto"
)

print("INT8 model loaded!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

INT8 model loaded!


In [10]:
os.makedirs("/content/quantized/model-int8", exist_ok=True)

model_int8.save_pretrained("/content/quantized/model-int8")
tokenizer.save_pretrained("/content/quantized/model-int8")

print("INT8 model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INT8 model saved!


In [11]:
from transformers import BitsAndBytesConfig

bnb_int4 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_int4 = AutoModelForCausalLM.from_pretrained(
    "/content/merged_model",
    quantization_config=bnb_int4,
    device_map="auto"
)

print("INT4 model loaded!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

INT4 model loaded!


In [12]:
os.makedirs("/content/quantized/model-int4", exist_ok=True)

model_int4.save_pretrained("/content/quantized/model-int4")
tokenizer.save_pretrained("/content/quantized/model-int4")

print("INT4 model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INT4 model saved!


In [13]:
!git clone https://github.com/ggerganov/llama.cpp.git
!pip install -r llama.cpp/requirements.txt -q
print("llama.cpp ready!")

Cloning into 'llama.cpp'...
remote: Enumerating objects: 81138, done.
remote: Counting objects: 100% (202/202), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 81138 (delta 163), reused 127 (delta 126), pack-reused 80936 (from 3)
Receiving objects: 100% (81138/81138), 302.36 MiB | 15.57 MiB/s, done.
Resolving deltas: 100% (58657/58657), done.
Updating files: 100% (2278/2278), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 14.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.2/96.2 kB 11.0 MB/s eta 0:00

In [16]:
!python /content/llama.cpp/convert_hf_to_gguf.py /content/merged_model \
    --outfile /content/quantized/model.gguf \
    --outtype q8_0

print("GGUF conversion done!")

INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> Q8_0, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> Q8_0, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> Q8_0, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> Q8_0, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> Q8_0, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> Q8_0, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.attn_out

In [15]:
from huggingface_hub import hf_hub_download
import shutil, os

tokenizer_model = hf_hub_download(
    repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    filename="tokenizer.model"
)

shutil.copy(tokenizer_model, "/content/merged_model/tokenizer.model")

print(os.listdir("/content/merged_model"))

['tokenizer.model', 'config.json', 'tokenizer_config.json', 'generation_config.json', 'chat_template.jinja', 'model.safetensors', 'tokenizer.json']


In [17]:
import os

def folder_size(path):
    total = sum(os.path.getsize(os.path.join(r, f))
                for r, _, files in os.walk(path)
                for f in files)
    return round(total / (1024 * 1024 * 1024), 2)

fp16_size = folder_size("/content/merged_model")
int8_size = folder_size("/content/quantized/model-int8")
int4_size = folder_size("/content/quantized/model-int4")
gguf_size = round(os.path.getsize("/content/quantized/model.gguf") / (1024 * 1024 * 1024), 2)

print(f"FP16 size: {fp16_size} GB")
print(f"INT8 size: {int8_size} GB")
print(f"INT4 size: {int4_size} GB")
print(f"GGUF size: {gguf_size} GB")

FP16 size: 2.05 GB
INT8 size: 1.15 GB
INT4 size: 0.76 GB
GGUF size: 1.09 GB


In [32]:
def benchmark(model, tokenizer, name):
    prompt = "Write Python code only, no explanation:\ndef add_two_numbers(a, b):\n     returns sum of a and b"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    start = time.time()
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        eos_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.2
    )
    end = time.time()
    tokens = outputs.shape[1]
    speed = round(tokens / (end - start), 2)
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\n=== {name} ===")
    print(f"Speed  : {speed} tokens/sec")
    print(f"Output : {text}")
    return speed, text

fp16_speed, fp16_output = benchmark(merged_model, tokenizer, "FP16")
int8_speed, int8_output = benchmark(model_int8, tokenizer, "INT8")
int4_speed, int4_output = benchmark(model_int4, tokenizer, "INT4")


=== FP16 ===
Speed  : 32.13 tokens/sec
Output : Write Python code only, no explanation:
def add_two_numbers(a, b):
     returns sum of a and b 

# Example usage
print(add_two_numbers(10,2)) # Outputs 12

=== INT8 ===
Speed  : 15.34 tokens/sec
Output : Write Python code only, no explanation:
def add_two_numbers(a, b):
     returns sum of a and b 
     
# Example usage
print(add_two_numbers(10,2)) # Outputs 12

=== INT4 ===
Speed  : 45.17 tokens/sec
Output : Write Python code only, no explanation:
def add_two_numbers(a, b):
     returns sum of a and b 

# Example usage
print(add_two_numbers(10,2)) # Output: 30


In [24]:
!pip install llama-cpp-python -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 18.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.7 MB/s eta 0:00:00


In [25]:
from llama_cpp import Llama
import time

llm = Llama(
    model_path="/content/quantized/model.gguf",
    n_gpu_layers=-1,
    verbose=False
)

start = time.time()
output = llm.create_chat_completion(
    messages=[{"role": "user", "content": "Write Python code only, no explanation:\ndef add_two_numbers(a, b):\n    # returns sum of a and b"}],
    max_tokens=100
)
end = time.time()

tokens = output['usage']['completion_tokens']
gguf_speed = round(tokens / (end - start), 2)
gguf_output = output['choices'][0]['message']['content']

print(f"\n=== GGUF ===")
print(f"Speed  : {gguf_speed} tokens/sec")
print(f"Output : {gguf_output}")

llama_context: n_ctx_per_seq (512) < n_ctx_train (2048) -- the full capacity of the model will not be utilized



=== GGUF ===
Speed  : 4.93 tokens/sec
Output : def add_two_numbers(a, b):
    # Returns the sum of a and b
    return a + b

# Example usage
print(add_two_numbers(10, 20)) # Output: 30

# Output: 30
# Example usage
print(add_two_numbers(10, 20)) # Output: 30
